In [2]:
import re
import json
import random
import numpy as np
import pandas as pd
# from sentence_transformers import SentenceTransformer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 1. save the database from the Googlesheet
# 2. upload the dataset to /Users/vivianzhang/edge&ease/Edge-and-Ease/backend
# 3. run the code below

In [ ]:
df = pd.read_excel(r'/Users/vivianzhang/edge&ease/Edge-and-Ease/backend/dataset.xlsx', dtype = str)

In [14]:
df

,Timestamp,Your Name,Title/Coaching Level,Coaching since which year,Are you comfortable with your name being displayed on the platform?,A skater feeling nervous before warm-up,A skater overwhelmed right before stepping on the ice.,A first-time competitor experiencing anxiety,A skater worrying about disappointing their coach or parents,A skater competing at a new level for the first time,...,A skater who worries about stamina,A skater disappointed after finishing their skate,A skater who fell or messed up during warm-up,A skater who skates last in the event,A skater who skates first in the event,A skater struggling with confidence,A skater distracted by the crowd,A skater who tends to overthink,A skater who has been having a bad training week,What is one piece of advice you consistently give competitive skaters that you believe has the greatest impact?
0,2026-02-23 17:00:49.846000,Leah Krauskopf,University of Delaware Collegiate Team Head Coach,2020,"Yes, full name",Nerves are normal since they mean that you car...,"Break down your program, talk abt each element...",Talk them through what to expect. Remind them ...,This is your journey and we are here every ste...,Remind them that it's your first year in level...,...,Let ur training speak for itself. If you have ...,Ask them how it felt and why are you disappoin...,This is a good thing since you can get it out ...,Don't worry about your order since order won't...,This is great because you get to set the tone ...,You are trained and ready. Nothing is differen...,Focus and worry about what you're doing. Don't...,Focus on specific things you need to do. Go th...,"Training is always gonna have ups and downs, w...",Trust yourself and just go out and enjoy it. R...
1,2026-02-24 10:19:04.112000,Anastasia Cannuscio,National and International coach,2010,"Yes, full name",Feeling nervous is completely normal and under...,"Close your eyes and take slow, deep breaths. T...","Feeling anxious is expected, but I am so confi...",Your coach and parents only want you to do you...,All of the work has already been done behind t...,...,You have done so many run-throughs leading up ...,Your feelings of being upset are completely va...,It's very normal to feel rattled after a mista...,"Don't watch your other competitors, go into a ...",Prioritize the elements you want to do on your...,"Confidence has to come from you, but when you'...",Focus on you and the work you have to do. Pick...,Try to visualize and focus on one element at a...,"One week does not define you, you have had man...",Before you even get to the competition the har...
2,2026-02-27 22:32:00.863000,Anna Lewis,Figure skating coach; National Senior Solo Dan...,2021 private around 2023,"Yes, full name","Trust your training, believe in yourself, and ...","Take a deep breath, take everything one thing ...","The stakes are low, so go out there and have f...","Don't worry, focus on how you're skating, and ...","Have an open mind, think positive thoughts, an...",...,"Take a deep breath, you just have to do what y...",Congratulations! It's okay to feel disappointe...,"Take a deep breath, it's in the past. It doesn...",Skating last because of placement: It's a priv...,"You can't control it, so just take easier warm...",Trust your practice and try your very best. Th...,Try to stay present and focus on your breath a...,Fake it till you make it because confidence is...,There's good weeks and bad weeks and good days...,Fake it till you make it. Everyone may seem ve...
3,2026-03-01 22:50:41.176000,Jon Nichols,US Junior National and National Coach in Ice D...,2008,"Yes, full name",Rely on your day-to-day training. It will be c...,We all feel like that. Rely on your everyday p...,It's a completely normal feeling. It gets easi...,"I understand it's a fear, but I can't imagine ...",It's easier said than done. I wouldn't have mu...,...,You can’t really make adjustments at that poin...,Even the best skaters in the world can be disa...,You really have to comp

In [15]:
col = df.columns.tolist()
col

['Timestamp',
 'Your Name',
 'Title/Coaching Level',
 'Coaching since which year',
 'Are you comfortable with your name being displayed on the platform?',
 'A skater feeling nervous before warm-up',
 'A skater overwhelmed right before stepping on the ice.',
 'A first-time competitor experiencing anxiety',
 'A skater worrying about disappointing their coach or parents',
 'A skater competing at a new level for the first time',
 'A skater worried about scores or judging',
 'A skater comparing themselves to other competitors',
 'A skater who made a mistake mid-program',
 'A skater who worries about stamina',
 'A skater disappointed after finishing their skate',
 'A skater who fell or messed up during warm-up',
 'A skater who skates last in the event',
 'A skater who skates first in the event',
 'A skater struggling with confidence',
 'A skater distracted by the crowd',
 'A skater who tends to overthink',
 'A skater who has been having a bad training week',
 'What is one piece of advice you

In [23]:
question_cols = df.columns[5:-1]
question_cols

Index(['A skater feeling nervous before warm-up',
       'A skater overwhelmed right before stepping on the ice.',
       'A first-time competitor experiencing anxiety',
       'A skater worrying about disappointing their coach or parents',
       'A skater competing at a new level for the first time',
       'A skater worried about scores or judging',
       'A skater comparing themselves to other competitors',
       'A skater who made a mistake mid-program',
       'A skater who worries about stamina',
       'A skater disappointed after finishing their skate',
       'A skater who fell or messed up during warm-up',
       'A skater who skates last in the event',
       'A skater who skates first in the event',
       'A skater struggling with confidence',
       'A skater distracted by the crowd', 'A skater who tends to overthink',
       'A skater who has been having a bad training week'],
      dtype='object')

In [24]:
df_long = df.melt(
    id_vars = ["Your Name", "Title/Coaching Level"], # If you need new keys, you can add them here.
    value_vars = question_cols,
    var_name = "question",
    value_name = "answer"
)

df_long

,Your Name,Title/Coaching Level,question,answer
0,Leah Krauskopf,University of Delaware Collegiate Team Head Coach,A skater feeling nervous before warm-up,Nerves are normal since they mean that you car...
1,Anastasia Cannuscio,National and International coach,A skater feeling nervous before warm-up,Feeling nervous is completely normal and under...
2,Anna Lewis,Figure skating coach; National Senior Solo Dan...,A skater feeling nervous before warm-up,"Trust your training, believe in yourself, and ..."
3,Jon Nichols,US Junior National and National Coach in Ice D...,A skater feeling nervous before warm-up,Rely on your day-to-day training. It will be c...
4,Brooklee Han,National and International Singles and Solo Da...,A skater feeling nervous before warm-up,Have a game plan going into this and stick wit...
...,...,...,...,...
165,Shira Selis,"Figure skating coach- skating skills, freestyl...",A skater who has been having a bad training week,You are mentally and physically prepared for t...
166,Pamela Gregory,Olympic and World Coach,A skater who has been having a bad training week,"As long as your training prior is solid, a few..."
167,Annie Luong,Figure Skating Coach and Choreographer,A skater who has been having a bad training week,Use this as a learning opportunity: timing doe...
168,Evan Bertz,National Junior Solo Dance & International Pat...,A skater who has been having a bad training week,The week leading up to the competition is alwa...


In [ ]:
df_long = df_long.reset_index(drop=True)
df_long["id"] = df_long.index + 1

result = df_long[["id", "Your Name", "Title/Coaching Level", "question", "answer"]]

json_data = result.to_dict(orient="records")

# save the json file
with open("qa_dataset.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)

In [34]:
result

,id,Your Name,Title/Coaching Level,question,answer
0,1,Leah Krauskopf,University of Delaware Collegiate Team Head Coach,A skater feeling nervous before warm-up,Nerves are normal since they mean that you car...
1,2,Anastasia Cannuscio,National and International coach,A skater feeling nervous before warm-up,Feeling nervous is completely normal and under...
2,3,Anna Lewis,Figure skating coach; National Senior Solo Dan...,A skater feeling nervous before warm-up,"Trust your training, believe in yourself, and ..."
3,4,Jon Nichols,US Junior National and National Coach in Ice D...,A skater feeling nervous before warm-up,Rely on your day-to-day training. It will be c...
4,5,Brooklee Han,National and International Singles and Solo Da...,A skater feeling nervous before warm-up,Have a game plan going into this and stick wit...
...,...,...,...,...,...
165,166,Shira Selis,"Figure skating coach- skating skills, freestyl...",A skater who has been having a bad training week,You are mentally and physically prepared for t...
166,167,Pamela Gregory,Olympic and World Coach,A skater who has been having a bad training week,"As long as your training prior is solid, a few..."
167,168,Annie Luong,Figure Skating Coach and Choreographer,A skater who has been having a bad training week,Use this as a learning opportunity: timing doe...
168,169,Evan Bertz,National Junior Solo Dance & International Pat...,A skater who has been having a bad training week,The week leading up to the competition is alwa...


In [37]:
json_data

[{'id': 1,
  'Your Name': 'Leah Krauskopf',
  'Title/Coaching Level': 'University of Delaware Collegiate Team Head Coach',
  'question': 'A skater feeling nervous before warm-up',
  'answer': 'Nerves are normal since they mean that you care and want to do well. Use them to fuel a good performance.'},
 {'id': 2,
  'Your Name': 'Anastasia Cannuscio',
  'Title/Coaching Level': 'National and International coach',
  'question': 'A skater feeling nervous before warm-up',
  'answer': 'Feeling nervous is completely normal and understandable. Use the warm-up to settle in and feel comfortable. Always remember you are prepared and to trust your training. '},
 {'id': 3,
  'Your Name': 'Anna Lewis',
  'Title/Coaching Level': 'Figure skating coach; National Senior Solo Dance Medalist; Solo International Pattern Dance National Champion',
  'question': 'A skater feeling nervous before warm-up',
  'answer': 'Trust your training, believe in yourself, and just try to be positive.'},
 {'id': 4,
  'Your Na

In [ ]:
df_ = df_long[['question']]
df_["id"] = df_.index + 1
df_["id"] = df_["id"].astype(str)
df_g = df_.groupby('question', dropna=False)['id'].agg(
    lambda x: "|".join(list(x))
).reset_index()
json_data_ = df_g.drop_duplicates().reset_index(drop=True).to_dict(orient="records")
json_data_

# another json
with open("only_qa_dataset.json", "w", encoding="utf-8") as f:
    json.dump(json_data_, f, ensure_ascii=False, indent=2)

C:\Users\sfeng\AppData\Local\Temp\ipykernel_21328\3885120081.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_["id"] = df_.index + 1
C:\Users\sfeng\AppData\Local\Temp\ipykernel_21328\3885120081.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_["id"] = df_["id"].astype(str)


## old code

don't need to run this

In [ ]:
def prepare_dor_rag(inputs: str):

    # 1. we need to GET the data from the frontend.

    # 2. clean it.
    # 2.1 hi                                                                     , I ......
    # 2.2 1999999999999999999, detect the digits return the error ().
    # 2.3 exceed your limit.
    # 2.4 only symbols
    # 2.5 other, error

    # 3. Embedding
    # 3.1 their inputs
    # 3.2 questions and coaches' names?

    # 4. cosine similarity
    # # return ID and score

    # 5. sort by score and find the final results.

    # 6. optional, collect the feedback.

    # 7. return results
    # 7.1 POST the results to the frontend.

    # 8. deployment
    # 8.1 save the main.py
    # 8.2 git push
    # refresh the website

    pass


In [ ]:
inputs = "Lower the brightness on your screen, put on soft ambient sound, and give yourself permission to work at 80% pace for the next 20 minutes."
input_dictionary = {'input':inputs}
limits = 1000
input_dictionary

{'input': 'Lower the brightness on your screen, put on soft ambient sound, and give yourself permission to work at 80% pace for the next 20 minutes.'}

In [45]:
# 2. clean it.
# 2.1 hi                                                                     , I ......
# 2.2 1999999999999999999, detect the digits return the error ().
# 2.3 exceed your limit.
# 2.4 only symbols
# 2.5 other, error

# 3. Embedding
# 3.1 their inputs
# 3.2 questions and coaches' names?

# 4. cosine similarity
# # return ID and score

# 5. sort by score and find the final results.

# 6. optional, collect the feedback.

# 7. return results
# 7.1 POST the results to the frontend.

def reach_limit(inputs: str, 
                max_len: int = limits):
    
    if len(inputs) > max_len:
        return True
    else:
        return False

def clean_text_column(input_dictionary: dict, null_value=np.nan) -> dict:
    """
    Refined text cleaner: strips, collapses, and flags issues.
    """
    out = input_dictionary.copy()
    raw_val = out.get('input')

    # 1. Handle actual Nulls immediately
    if pd.isna(raw_val):
        out["cleaned_text"] = null_value
        out["issue_type"] = "empty_error"
        return out

    # Ensure string type for processing
    text = str(raw_val)
    
    # 2. Check for "Extra Spaces" before we wipe them
    # (Leading, trailing, or double spaces)
    had_extra_spaces = bool(re.search(r"^\s|\s$|\s{2,}", text))

    # 3. Apply basic normalization (Strip and Collapse)
    cleaned = re.sub(r"\s+", " ", text).strip()

    # 4. Define Flags
    is_empty = cleaned == ""
    is_digits = bool(re.fullmatch(r"\d+", cleaned))
    is_symbols = bool(re.fullmatch(r"[^\w\s]+", cleaned))
    has_bad_chars = bool(re.search(r"[\x00-\x1F\x7F]", cleaned))

    # 5. Determine Issue Type (Priority Order)
    if is_empty:
        issue = "empty_error"
        final_text = null_value
    elif is_digits:
        issue = "digits_only_error"
        final_text = null_value
    elif is_symbols:
        issue = "symbols_only_error"
        final_text = null_value
    elif has_bad_chars:
        issue = "other_error"
        final_text = null_value
    elif had_extra_spaces:
        issue = "extra_space_cleaned"
        final_text = cleaned
    else:
        issue = "valid"
        final_text = cleaned

    out["cleaned_text"] = final_text
    out["issue_type"] = issue

    return out

In [ ]:
qa_json_path = r'/Users/vivianzhang/edge&ease/Edge-and-Ease/backend/qa_dataset.json'

In [ ]:
def embedding_and_similarity(input_dictionary: dict):
    vectorizer = TfidfVectorizer()

    # user_inputs = ["I need help with leadership", "How to manage stress?"]
    user_query = input_dictionary.get('input', "")

    # OPTIONAL --------------------- find the coach name

    # with open(qa_json_path, "r", encoding="utf-8") as f:
    #     qa_json = json.load(f)
    
    reference_data = [
                    {
                        "question": "A first-time competitor experiencing anxiety",
                        "id": "21|22|23|24|25|26|27|28|29|30"
                    },
                    {
                        "question": "A skater comparing themselves to other competitors",
                        "id": "61|62|63|64|65|66|67|68|69|70"
                    },
                    {
                        "question": "A skater competing at a new level for the first time",
                        "id": "41|42|43|44|45|46|47|48|49|50"
                    },
                    {
                        "question": "A skater disappointed after finishing their skate",
                        "id": "91|92|93|94|95|96|97|98|99|100"
                    }]

    reference_data = [
        {"id": 101, "text": "Executive Leadership Coaching", "type": "coach_name"},
        {"id": 102, "text": "How do I improve my team management?", "type": "question"},
        {"id": 103, "text": "Dr. Sarah Wellness", "type": "coach_name"},
        {"id": 104, "text": "Strategies for reducing workplace stress", "type": "question"}
    ]

    # Extract texts for embedding
    ref_questions = [item['question'] for item in reference_data]
    # ["A first-time competitor experiencing anxiety", 
    # "A skater comparing themselves to other competitors", 
    # "A skater competing at a new level for the first time", 
    # "A skater disappointed after finishing their skate"]

    ref_ids = [item['id'].split("|") for item in reference_data]
    # [[21, 22, 23, 24, 25, 26, 27, 28, 29, 30], 
    # [61, 62, 63, 64, 65, 66, 67, 68, 69, 70], 
    # [41, 42, 43, 44, 45, 46, 47, 48, 49, 50], 
    # [91, 92, 93, 94, 95, 96, 97, 98, 99, 100]]

    ref_questions = [item['question'] for item in reference_data]
    ref_ids = [item['id'] for item in reference_data]

    vectorizer = TfidfVectorizer()
    
    ref_matrix = vectorizer.fit_transform(ref_questions)
    query_vec = vectorizer.transform([user_query])

    similarity_scores = cosine_similarity(query_vec, ref_matrix).flatten()

    #  Find the best match
    best_idx = np.argmax(similarity_scores)
    highest_score = similarity_scores[best_idx]

    possible_ids = ref_ids[best_idx]  # Get the list, e.g., [21, 22, 23...]
    selected_id = random.choice(possible_ids)

    with open("qa_dataset.json", "r", encoding="utf-8") as f:
        qa_json = json.load(f)

    answer = qa_json[selected_id]

    # Build Result
    result = {
        "user_query": user_query,
        "matched_question": ref_questions[best_idx],
        "random_selected_id": selected_id,
        "answer": answer,
        "all_possible_ids": str(possible_ids), # Optional: keep for debugging
        "score": round(float(highest_score), 4)
    }
    print("🥉-------------------The result is-------------------")
    print(pd.DataFrame([result]))

    return result